In [0]:
%run "../libs/libs"

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col, to_date

In [0]:
invoice_df = spark.read.format("delta").table("dev.tracking_sl.invoice")
org_df = spark.read.format("delta").table("dev.tracking_gd.dimorganization")
diminvoice_df = spark.read.format("delta").table("dev.tracking_gd.diminvoice")

In [0]:
customer = org_df.alias('customer')
invoice = invoice_df.alias('invoice')
diminvoice = diminvoice_df.alias('diminvoice')

tdetail_df = (
    invoice.join(
        customer,
        (invoice.nameofcustomer == customer['name']) &
        (invoice.invoicecity == customer['city']) &
        (invoice.invoicestate == customer['state']),
        "inner"
    )  
    .join(
        diminvoice,
        (invoice.invoiceno == diminvoice['invoiceno']),
        "inner"
    )     
.select(
        col('datekey'),
        col('invoiceid').alias('invoice_id'),
        col('invoiceamount').alias('invoice_amt'),
        col('taxrate').alias('tax_rate'),
        col('taxamount').alias('tax_amount'),
        col('paymentdays'),
        col('orgid').alias('customer_id')
    )
    .withColumn('ingestion_time', current_timestamp())
)

In [0]:
tdetail_df.createOrReplaceTempView("temp_factinvoice")

In [0]:
%sql
merge into dev.tracking_gd.factinvoice tgt
USING ( 
  SELECT DISTINCT * 
  FROM temp_factinvoice 
) src 
on tgt.invoice_id = src.invoice_id
when not matched then
  insert (datekey, invoice_id, invoice_amt,tax_rate,tax_amount,paymentdays,customer_id, ingestion_time)
  Values(src.datekey,src.invoice_id,src.invoice_amt,src.tax_rate,src.tax_amount,src.paymentdays,src.customer_id,src.ingestion_time)